# Automejora del prompt

[Guía de la entrega](../../../../docs/README.md) · [Caso e historial](../README.md)

**Objetivo:** mejorar automáticamente las instrucciones que usa el agente para negociar una deuda.

**H (política):** instrucciones para negociar. **φ (instrucciones de mejora):** texto que guía cómo proponer cambios en esa política.

![Objetivo y flujo](../diagrams/objective-flow.svg)

| Estrategia | H (política) | φ (instrucciones de mejora) |
|---|---|---|
| **B1: mejora fija** | Puede cambiar | Se mantiene igual |
| **B2: mejora adaptativa** | Puede cambiar | Puede reemplazarse entre intentos |

**Restricción:** mismo modelo Qwen3 8B a 4 bits y pesos sin entrenar durante la comparación.
H0 y H1 son versiones del texto; B1 y B2 son estrategias. Una versión nueva no garantiza mejores resultados.

Las celdas leen ejecuciones guardadas. No generan respuestas nuevas ni consumen API.

## 1. Caso y respuesta

**Entrada del negociador:**

- **Tabla de deudas:** montos ficticios y planes permitidos, indicados abajo.
- **Historial de conversación:** mensajes previos del cliente y del asistente. Puede incluir capacidad de pago, ofertas, rechazo o aceptación; no es un historial de pagos.

El agente elige una acción: **preguntar** por la capacidad (`ask`), **ofrecer** un plan (`offer`),
**confirmar** un plan aceptado (`accept`) o **detenerse** (`stop`).

| Dato del ejemplo | Valor |
|---|---|
| Mensaje ilustrativo del cliente | «Puedo destinar 225 al mes para pagar». |
| Capacidad de pago mensual | **225 unidades ficticias** disponibles para la deuda; no es el ingreso mensual |
| Planes permitidos | Un pago o dos cuotas mensuales iguales |

| Deuda | Monto total / un pago | Cuota con dos pagos | Evaluación |
|---|---:|---:|---|
| A | 200 | 100 | Viable, pero de menor monto |
| B | 300 | 150 | **Mayor deuda viable: ofrecer dos cuotas** |
| C | 600 | 300 | La cuota supera la capacidad de pago |

**JSON esperado del negociador** en los Tests 3–5:

```json
{
  "action": "offer",
  "debt": "B",
  "payments": 2,
  "installment": 150
}
```

| Campo | Significado |
|---|---|
| `action` | Acción; `offer` significa ofrecer, no confirmar un acuerdo |
| `debt` | Deuda elegida; `B` identifica el monto de 300 |
| `payments` | Número de pagos; aquí, dos |
| `installment` | Monto de cada cuota mensual; aquí, 150 |

Para `ask` y `stop`: `debt=null`, `payments=0`, `installment=0`.
Python redacta el mensaje a partir de esos campos, sin corregir la decisión.
La respuesta esperada pertenece al evaluador: **el negociador no la recibe**.

## 2. Pruebas ejecutadas

| Etapa | Pregunta | Registro |
|---|---|---|
| Piloto | ¿Funciona una búsqueda inicial con mejorador fijo? | [Piloto](prompt-search.ipynb) |
| Test 1 | ¿Se actualiza y reutiliza φ? | [Adaptación mínima](harness-smoke-test.ipynb) |
| Test 2 | ¿Pueden 4B y 8B resolver la tarea? | [Modelos locales](local-model-test.ipynb) |
| Test 3 | ¿Mejora el negociador al ajustar inferencia? | [Inferencia](inference-tuning.ipynb) |
| Test 4 | ¿Qué aporta adaptar φ frente a mantenerla fija? | [B1/B2](paired-search.ipynb) |
| Test 5 | ¿Restringir JSON mejora las decisiones? | [Formato](structured-output.ipynb) |

El Test 3 separa capacidad del negociador y calidad del mejorador. Esta lectura desarrolla
los Tests 3–5; los enlaces conservan los protocolos completos, incluidos intentos fallidos.

**Modelos registrados:** Qwen3 4B, 8B y 32B, cuantizados a 4 bits. B indica miles de millones de parámetros; 4 bits indica la representación de los pesos. Los controles API usan GPT-4o mini.

In [1]:
from pathlib import Path
import sys

relative = Path("research/cases/case-0003-debt-negotiation")
case_dir = next(base / relative for base in [Path.cwd(), *Path.cwd().parents]
                if (base / relative / "results").is_dir())
sys.path.insert(0, str(case_dir / "utils"))
import study_view as view

records = view.load_records(case_dir)
print(f"Evidencia verificada: {len(records)} ejecuciones. Sin llamadas al modelo.")

Evidencia verificada: 8 ejecuciones. Sin llamadas al modelo.


## 3. Ajuste de inferencia

Se ajusta **cómo el modelo genera respuestas**, sin entrenarlo. Un token es una unidad de texto del modelo; no equivale siempre a una palabra.

| Parámetro | Significado | Valores y fundamento |
|:---|:---|:---|
| Temperatura | Regula la diversidad al elegir el siguiente token | **0:** elegir el más probable.<br>**0,3:** explorar poca variación.<br>**0,7:** referencia sin razonamiento.<br>**0,6:** referencia con razonamiento. |
| `top_k` | Limita la selección a los k tokens más probables | **20:** referencia de Qwen; fijo durante el barrido. |
| `top_p` | Filtra candidatos por probabilidad acumulada | **0,8:** sin razonamiento.<br>**0,95:** con razonamiento. |
| `max_tokens` | Tope de salida por llamada, incluido razonamiento | **512:** barrido directo.<br>**2.048:** con razonamiento.<br>**4.096:** comprobar cortes de salida. |

**Referencia:** [Qwen3](https://huggingface.co/Qwen/Qwen3-8B#best-practices). Temperaturas 0 y 0,3: exploratorias. Topes de tokens: límites locales de prueba.

Un **escenario** contiene una conversación simulada y su decisión esperada.

| Grupo del Test 3 | Uso |
|:---|:---|
| **12 conocidos** | Conversaciones ya usadas en diagnóstico.<br>Incluyen capacidad de pago, planes, rechazo y consentimiento.<br>Permiten comparar perfiles; no entrenan pesos. |
| **8 reservados** | Conversaciones distintas, fijadas antes del barrido.<br>No intervienen en la selección del perfil.<br>Permiten comprobarlo después. |
| **3 repeticiones** | Misma conversación y perfil; distintas semillas.<br>**8 casos × 3 repeticiones = 24 respuestas.**<br>Son ocho casos, no 24 casos diferentes. |

- **Semilla:** inicializa el muestreo aleatorio para repetir la configuración.
- **Valores:** 11, 22 y 33; identifican repeticiones, no se optimizaron.
- **Uso posterior:** los ocho casos se reutilizaron en otros diagnósticos; ya no eran una prueba nueva.

Sea **N** el número de respuestas evaluadas:

$$Q=\frac{\text{respuestas correctas sin infracciones}}{N},\qquad
V=\frac{\text{respuestas con al menos una infracción}}{N}.$$

- **Respuesta correcta:** coincide con la referencia en:
  - Acción: preguntar, ofrecer, aceptar o detenerse.
  - Deuda elegida.
  - Cantidad de pagos.
  - Monto de cada cuota.

- **Infracciones:** se registra alguna de estas condiciones:
  - Salida inválida o incompleta.
  - Plan no permitido.
  - Cuota mal calculada o superior a la capacidad de pago.
  - Oferta tras un rechazo explícito a negociar.
  - Aceptación sin consentimiento.

- **Q ≥ 80 %:** respuestas correctas y sin infracciones:
  - Con 8 respuestas: al menos **7 aciertos**.
  - Con 24 respuestas: al menos **20 aciertos**.

- **V = 0:** ninguna respuesta evaluada infringe las reglas.
  - Una cuota impagable o una aceptación inventada invalida la decisión.

**Ejemplo:** ofrecer A cuando B también cabe:
- La elección es incorrecta: reduce Q.
- El plan puede respetar las restricciones: no necesariamente aumenta V.

**Alcance:** 80 % es una meta provisional del piloto. V = 0 describe las respuestas evaluadas, no todos los casos posibles.

![Control de inferencia](../diagrams/inference-control.svg)

**Control:** H manual fija, sin mejorador ni actualizador. Esta política no se entrega a la búsqueda de los Tests 4–5.

In [2]:
view.prompt("Test 3 · H manual fija", records["inference"]["protocol"]["policy"], folded=True)
rows = []
for key, title in [("inference", "Ajuste: conocidos"), ("confirmation", "Confirmación: reservados")]:
    report = records[key]["report"]
    samples = [row for evaluation in report["evaluations"] for row in evaluation["rows"]]
    rows.append([title, f"{sum(row['success'] for row in samples)}/{len(samples)}",
                 sum(bool(row["violations"]) for row in samples), report["tokens"]])
view.table(["Etapa", "Aciertos", "Respuestas con infracciones", "Tokens totales"], rows)

Etapa,Aciertos,Respuestas con infracciones,Tokens totales
Ajuste: conocidos,12/12,0,15110
Confirmación: reservados,24/24,0,31036


## 4. Prompts y roles

- **System prompt:** instrucciones para una llamada. Combina un contrato fijo (límites y formato) con el texto editable H o φ, según el rol.
- **Entrada `user`:** datos para esa llamada. En el negociador son deudas y conversación; en el mejorador son objetivo, H y fallos de desarrollo.
- **Salida:** decisión JSON del negociador, nueva H del mejorador o nueva φ del actualizador.

**Ejemplos ilustrativos:** H: «Ofrece la mayor deuda viable». φ: «Revisa primero los fallos de capacidad y corrige esa regla».
φ guía al mejorador; **no se añade al prompt del negociador**. Una versión aceptada reemplaza a la anterior.

![Entradas y salidas](../diagrams/model-inputs.svg)

Los desplegables muestran los system prompts completos de la ejecución. El mejorador recibe respuestas y referencias de desarrollo; nunca los casos reservados para el test final.

In [3]:
view.show_start(records)

## 5. Primera mejora

![Mejora del harness](../diagrams/learning-loop.svg)

$$\widetilde H_1=M_L(H_0,\phi_0,O,F_{dev}(H_0))$$

- **M:** mejorador, ejecutado con el modelo **L**.
- **O:** objetivo fijo del caso.
- **Fdev:** respuestas, aciertos y fallos de desarrollo.
- **H̃1:** política candidata. Se convierte en H1 conservada solo si pasa la selección.

**Selección:** mejora de Q en validación ≥ 0,01 y V = 0. En empate se conserva H.
No se suman prompts: H aceptada reemplaza la política anterior.

In [4]:
view.show_first_proposal(records)

Caso de desarrollo,Respuesta de H0,Referencia,Correcta,Incumplimientos
Elegir la mayor deuda viableCapacidad de pago: 330 al mesID: development-largest,Sin respuesta válida,"{ ""action"": ""offer"", ""debt"": ""C"", ""payments"": 2, ""installment"": 300 }",No,Corte por límite de salida (output_limit)
Respetar el rechazoEl cliente pide detener la negociaciónID: development-declined,"{ ""action"": ""stop"", ""debt"": null, ""payments"": 0, ""installment"": 0 }","{ ""action"": ""stop"", ""debt"": null, ""payments"": 0, ""installment"": 0 }",Sí,Ninguno
Preguntar por la capacidadEl cliente aún no sabe cuánto puede pagarID: transfer-unknown,"{ ""action"": ""ask"", ""debt"": null, ""payments"": 0, ""installment"": 0 }","{ ""action"": ""ask"", ""debt"": null, ""payments"": 0, ""installment"": 0 }",Sí,Ninguno
Confirmar el plan aceptadoEl cliente acepta A en dos cuotas de 100ID: confirmation-consent,"{ ""action"": ""accept"", ""debt"": ""A"", ""payments"": 2, ""installment"": 100 }","{ ""action"": ""accept"", ""debt"": ""A"", ""payments"": 2, ""installment"": 100 }",Sí,Ninguno


Versión,Q validación,V validación,Selección
H0,0%,0%,Reemplazada
H1,100%,0%,Aceptada


## 6. Adaptación de instrucciones

**B1:** φ0 (instrucciones iniciales de mejora) permanece fija. **B2:** el actualizador puede reemplazar esas instrucciones.

`φ1 = actualizador(φ0, objetivo, H candidata 1, fallos de desarrollo)`

La siguiente propuesta recibe **H (política conservada)** y **φ (instrucciones vigentes)**. Una candidata rechazada
no sustituye H. Una actualización de φ no demuestra por sí sola una mejora de calidad.

Las versiones siguientes muestran el texto real, no una paráfrasis.

In [5]:
view.show_updates(records)

Versión,Tras intento,Texto cambiado,Usada en intento,Error
φ1,1,Sí,2,Ninguno
φ2,2,Sí,3,Ninguno
φ3,4,Sí,5,Ninguno


## 7. Intentos y selección

Cada búsqueda tiene cinco intentos. Una salida inválida consume un intento; no se evalúa como política.
Una candidata duplicada reutiliza su puntuación. H solo cambia si supera a la vigente.

En la tabla, Q y V corresponden a **validación**, no al test final.

In [6]:
view.show_rounds(records)

Condición,Intento,Parte de,Q candidata,V candidata,Resultado,H conservada
B1,0,—,0%,0%,Inicio,H0
B1,1,H0,100%,0%,Aceptada,H1
B1,2,H1,75%,0%,Rechazada,H1
B1,3,H1,75%,0%,Duplicada del intento 2,H1
B1,4,—,Sin evaluar,Sin evaluar,Fuera del esquema; sin política evaluable,H1
B1,5,H1,75%,0%,Duplicada del intento 2,H1
B2,0,—,0%,0%,Inicio,H0
B2,1,H0,100%,0%,Aceptada,H1
B2,2,H1,75%,0%,Rechazada,H1
B2,3,—,Sin evaluar,Sin evaluar,Fuera del esquema; sin política evaluable,H1


## 8. Tamaño del prompt

$$T(H)=|\mathrm{tokenizer}(H)|,\qquad T(\phi)=|\mathrm{tokenizer}(\phi)|$$

- **Tokens:** conteos registrados con el tokenizer nativo, sin tokens especiales.
- **Palabras:** segmentos separados por espacios o saltos de línea: `len(text.split())`.
- **Caracteres:** longitud Unicode del texto, incluidos espacios: `len(text)`.

Tabla del **Test 4 con 2.048 tokens de salida**. H y φ se miden por separado. No incluyen contrato, esquema, historial ni plantilla de chat.
Los tokens totales de ejecución sí incluyen entradas completas, respuestas y razonamiento.

La longitud se registró, pero **no fue criterio de selección**. Si un intento no registra
el conteo, se muestra «No registrado», no cero. No se necesita repetir inferencia para leer estas medidas.

In [7]:
view.show_lengths(records)

Búsqueda · intento,Caracteres,Palabras,Tokens,Q validación
B1 · 0,123,20,28,0%
B1 · 1,404,63,107,100%
B1 · 2,478,69,131,75%
B1 · 3,478,69,131,75%
B1 · 5,478,69,131,75%
B2 · 0,123,20,28,0%
B2 · 1,404,63,107,100%
B2 · 2,472,71,130,75%
B2 · 4,566,83,159,75%
B2 · 5,711,113,235,75%


Búsqueda · intento,Caracteres,Palabras,Tokens
B1 · 0,136,21,31
B1 · 1,136,21,31
B1 · 2,136,21,31
B1 · 3,136,21,31
B1 · 4,136,21,No registrado
B1 · 5,136,21,31
B2 · 0,136,21,31
B2 · 1,136,21,31
B2 · 2,628,95,170
B2 · 3,698,103,No registrado


**Lectura de longitud:** H0 tiene 28 tokens; la H1 aceptada, 107. Las candidatas posteriores
son más largas y no superan Q de validación de H1. Esto no demuestra que más longitud cause
peores respuestas: también cambió el contenido. Para aislarlo se necesitan variantes equivalentes
en contenido y distintos tamaños.

## 9. Decisiones observadas

**Caso resuelto:** capacidad de pago mensual 225. La referencia es B, dos pagos de 150.
Se muestran el historial y las respuestas originales de H0 y H seleccionada.

In [8]:
view.show_case(records, "paired-225")

Rol,Historial original
user,Para estas deudas tengo un límite de 225 por mes. ¿Qué alternativa corresponde?


Versión,Acción,Deuda,Pagos,Cuota,Correcta
Referencia,offer,B,2,150,—
B1 · H0,ask,None,0,0,No
B1 · H1,offer,B,2,150,Sí
B2 · H0,ask,None,0,0,No
B2 · H1,offer,B,2,150,Sí


**Caso pendiente:** el cliente actualiza su capacidad de pago mensual a 125. Corresponde A en dos pagos de 100.
La respuesta `stop` respeta el formato, pero no resuelve la tarea.

In [9]:
view.show_case(records, "paired-125")

Rol,Historial original
user,Antes tenía 250 disponibles al mes.
assistant,¿Cuál es tu límite actual?
user,Ahora solo puedo comprometer 125 mensuales. Revisa las opciones con ese monto.


Versión,Acción,Deuda,Pagos,Cuota,Correcta
Referencia,offer,A,2,100,—
B1 · H0,ask,None,0,0,No
B1 · H1,stop,None,0,0,No
B2 · H0,ask,None,0,0,No
B2 · H1,stop,None,0,0,No


## 10. Tokens y formato

- **Test 4:** tope por llamada de 2.048 a 4.096 tokens. Mismo inicio, casos y límites totales.
- **Test 5:** 4.096 tokens y JSON restringido durante la generación.
- **Casos finales conocidos:** se reutilizan para diagnóstico; no son una nueva confirmación.

«Llamadas sin error» incluye los tres roles; «aciertos» solo mide decisiones del negociador.
Tener JSON válido no equivale a elegir el plan correcto.

In [10]:
view.show_outcomes(records)

Prueba · tope,Condición,H0: aciertos,H elegida: aciertos,V final,Llamadas sin error,Cortes,Tokens totales
Test 4 · 2.048,B1,3/8,H1: 7/8,0%,43/45,1,52511
Test 4 · 2.048,B2,3/8,H1: 7/8,0%,62/64,1,78624
Test 4 · 4.096,B1,3/8,H2: 7/8,0%,53/53,0,68762
Test 4 · 4.096,B2,3/8,H1: 7/8,0%,63/65,0,84475
Test 5 · 4.096,B1,3/8,H2: 7/8,0%,53/53,0,68762
Test 5 · 4.096,B2,3/8,H1: 7/8,0%,73/73,0,91579


## 11. Resultados y límites

- **Inferencia:** el perfil con razonamiento cumple en el control con política manual.
- **Automejora inicial (2.048):** B1 y B2 conservan H1; pasan de 3/8 a 7/8 decisiones correctas en test.
- **Adaptación:** B2 reutiliza tres cambios de φ, sin ventaja final en aciertos.
- **Salida:** 4.096 tokens elimina cortes; JSON restringido elimina errores de formato en las ejecuciones registradas. Se mantiene 7/8.
- **Longitud:** más tokens en las candidatas no implica mejor puntuación.

Una búsqueda por condición y pocos escenarios sintéticos. Falta repetir con casos nuevos
para estudiar estabilidad y generalización.